# Symptom-Topology-Guided Lightweight Adaptive Graph Neural Network
## Full implementation notebook

This notebook implements the workflow described in the supplied paper:

1. SympScan data loading
2. Unified binary preprocessing
3. Disease–symptom bipartite graph construction
4. Jaccard symptom-topology disease graph
5. Compact low-dimensional graph representation
6. Lightweight local graph learning with adaptive neighbour attention
7. Lightweight global context learning with low-rank attention
8. Adaptive local–global structural fusion
9. Structure-preserving graph contrastive refinement
10. Lightweight neural bilinear disease relationship inference
11. Validation-based adaptive thresholding and enhanced graph construction
12. Supplementary disease-resource mapping

The implementation is designed to run in Google Colab. It can use a local CSV/folder or download the public SympScan dataset when the Kaggle API is configured.


In [ ]:
# ============================================================
# 0. INSTALLATION
# ============================================================
!pip -q install pandas numpy scikit-learn networkx matplotlib seaborn openpyxl tqdm kaggle

import os, re, gc, math, random, json, warnings, hashlib, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    precision_recall_curve, roc_curve
)
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Environment ready.")


## 1. Dataset configuration

The paper specifies the Kaggle SympScan dataset as the primary input and reports 85 unique diseases, 172 symptoms and 758 disease–symptom relationships. The code below searches common Colab locations first and then optionally uses the Kaggle API.

If you already have the SympScan CSV, set `DATA_PATH` to it.


In [ ]:
# ============================================================
# 1. DATASET CONFIGURATION
# ============================================================

DATA_PATH = ""  # Example: "/content/drive/MyDrive/SympScan.csv"
WORK_DIR = Path("/content/symptom_topology_gnn")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Optional Google Drive mount
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("Google Drive available.")
except Exception:
    print("Not running in Google Colab or Drive is unavailable.")

def locate_csv():
    candidates = []
    if DATA_PATH:
        candidates.append(Path(DATA_PATH))

    roots = [
        Path("/content"),
        Path("/content/drive/MyDrive"),
        Path("/mnt/data")
    ]

    keywords = ["sympscan", "symptom", "disease"]
    for root in roots:
        if root.exists():
            for p in root.rglob("*.csv"):
                name = p.name.lower()
                if any(k in name for k in keywords):
                    candidates.append(p)

    seen = set()
    unique = []
    for p in candidates:
        try:
            p = p.resolve()
            if p not in seen and p.exists():
                unique.append(p)
                seen.add(p)
        except Exception:
            pass
    return unique

found = locate_csv()

if found:
    print("Candidate CSV files:")
    for i, p in enumerate(found[:20]):
        print(i, p)
else:
    print("No local SympScan-like CSV found.")
    print("If needed, configure Kaggle API and run the download cell below.")


In [ ]:
# ============================================================
# 2. OPTIONAL KAGGLE DOWNLOAD
# ============================================================
# The paper identifies:
# https://www.kaggle.com/datasets/behzadhassan/sympscan-symptomps-to-disease
#
# Run this cell only if the dataset is not already available.

USE_KAGGLE = False

if USE_KAGGLE:
    import subprocess, os, glob
    subprocess.run([
        "kaggle", "datasets", "download",
        "-d", "behzadhassan/sympscan-symptomps-to-disease",
        "-p", str(WORK_DIR), "--unzip"
    ], check=False)

    print("Downloaded files:")
    for p in WORK_DIR.rglob("*"):
        if p.is_file():
            print(p)


In [ ]:
# ============================================================
# 3. LOAD RAW SYMPSCAN DATA
# ============================================================

# Select DATA_PATH if supplied; otherwise select the first detected CSV.
if DATA_PATH:
    csv_path = Path(DATA_PATH)
else:
    candidates = [p for p in WORK_DIR.rglob("*.csv")]
    if not candidates:
        candidates = locate_csv()
    if not candidates:
        raise FileNotFoundError(
            "No dataset CSV found. Set DATA_PATH to your SympScan CSV."
        )
    csv_path = candidates[0]

raw_df = pd.read_csv(csv_path)

print("CSV:", csv_path)
print("Shape:", raw_df.shape)
display(raw_df.head())
print("\nColumns:")
print(raw_df.columns.tolist())


## 4. Unified binary preprocessing

The paper describes:

- hash-based duplicate disease-record detection
- exact-match symptom deduplication
- binary-mode imputation
- binary consistency validation
- disease-label normalization
- symptom string normalization
- null-record filtering
- column-wise duplicate detection
- no Robust/Min-Max scaling because the inputs are binary


In [ ]:
# ============================================================
# 4. PREPROCESSING
# ============================================================

def normalize_text(x):
    x = str(x).strip().lower()
    x = re.sub(r"[_\-]+", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x

def find_disease_column(df):
    preferred = [
        "disease", "Disease", "diseases", "Disease Name",
        "disease_name", "prognosis", "label", "Label"
    ]
    for c in preferred:
        if c in df.columns:
            return c

    # Fallback: first low-cardinality object column
    obj = df.select_dtypes(include=["object"]).columns.tolist()
    if not obj:
        raise ValueError("Could not identify disease column.")
    scores = [(c, df[c].nunique(dropna=True)) for c in obj]
    scores.sort(key=lambda x: x[1])
    return scores[0][0]

df = raw_df.copy()
disease_col = find_disease_column(df)

# Disease label normalization
df[disease_col] = df[disease_col].map(normalize_text)

# Null-record filtering
df = df.dropna(how="all").copy()
df = df[df[disease_col].notna() & (df[disease_col].astype(str).str.len() > 0)].copy()

# Hash-based duplicate row detection
row_hash = pd.util.hash_pandas_object(df.astype(str), index=False)
before = len(df)
df = df.loc[~row_hash.duplicated()].copy()
print("Duplicate full rows removed:", before - len(df))

# Identify symptom columns
symptom_cols = [c for c in df.columns if c != disease_col]

# Symptom string normalization
rename_map = {}
for c in symptom_cols:
    rename_map[c] = normalize_text(c)

df = df.rename(columns=rename_map)

# Exact-match duplicate symptom columns
df = df.loc[:, ~df.columns.duplicated()].copy()

# Re-identify symptom columns after renaming
disease_col = normalize_text(disease_col)
if disease_col not in df.columns:
    # Find normalized version
    possible = [c for c in df.columns if c == normalize_text(raw_df.columns[raw_df.columns.get_loc(
        next(c for c in raw_df.columns if c == raw_df.columns[0])
    )])]

symptom_cols = [c for c in df.columns if c != disease_col]

# Binary conversion
def binary_value(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().lower()
        if s in {"1","true","yes","y","present","positive"}:
            return 1
        if s in {"0","false","no","n","absent","negative",""}:
            return 0
        try:
            x = float(s)
        except Exception:
            return np.nan
    try:
        return 1 if float(x) > 0 else 0
    except Exception:
        return np.nan

for c in symptom_cols:
    df[c] = df[c].map(binary_value)

# Binary-mode imputation per symptom
for c in symptom_cols:
    mode = df[c].mode(dropna=True)
    fill = int(mode.iloc[0]) if len(mode) else 0
    df[c] = df[c].fillna(fill).astype(np.int8)

# Remove empty symptom columns
symptom_cols = [c for c in symptom_cols if df[c].sum() > 0]

# Column-wise duplicate detection
X_bin = df[symptom_cols].T
dup_mask = X_bin.duplicated(keep="first")
duplicate_symptom_cols = X_bin.index[dup_mask].tolist()

if duplicate_symptom_cols:
    df = df.drop(columns=duplicate_symptom_cols)
symptom_cols = [c for c in df.columns if c != disease_col]

# Collapse repeated disease records using maximum symptom presence
binary_df = df.groupby(disease_col, as_index=False)[symptom_cols].max()

# Final binary validation
for c in symptom_cols:
    binary_df[c] = (binary_df[c].astype(float) > 0).astype(np.int8)

# Remove all-zero symptom columns
symptom_cols = [c for c in symptom_cols if binary_df[c].sum() > 0]
binary_df = binary_df[[disease_col] + symptom_cols]

print("Cleaned shape:", binary_df.shape)
print("Unique diseases:", binary_df[disease_col].nunique())
print("Unique symptoms:", len(symptom_cols))
print("Total observed disease-symptom relationships:",
      int(binary_df[symptom_cols].to_numpy().sum()))

binary_df.to_csv(WORK_DIR / "clean_binary_symptom_disease_matrix.csv", index=False)
display(binary_df.head())


## 5. Incidence-matrix bipartite graph

Diseases and symptoms are represented as two node types. A binary value of 1 creates a disease–symptom edge.


In [ ]:
# ============================================================
# 5. BIPARTITE GRAPH
# ============================================================

diseases = binary_df[disease_col].tolist()
symptoms = symptom_cols

B = nx.Graph()

disease_nodes = [f"D::{d}" for d in diseases]
symptom_nodes = [f"S::{s}" for s in symptoms]

B.add_nodes_from(disease_nodes, node_type="disease")
B.add_nodes_from(symptom_nodes, node_type="symptom")

A_ds = binary_df[symptoms].to_numpy(dtype=np.float32)

for i, dnode in enumerate(disease_nodes):
    for j, snode in enumerate(symptom_nodes):
        if A_ds[i, j] > 0:
            B.add_edge(dnode, snode, weight=1.0)

print("Bipartite nodes:", B.number_of_nodes())
print("Bipartite edges:", B.number_of_edges())

nx.write_gexf(B, WORK_DIR / "symptom_disease_bipartite_graph.gexf")


## 6. Symptom-topology mathematical disease graph

For each disease pair, Jaccard similarity is calculated from their binary symptom neighbourhoods:

`J(A,B) = |A ∩ B| / |A ∪ B|`

Only pairs with non-zero overlap are connected. The Jaccard value is the disease relationship weight.


In [ ]:
# ============================================================
# 6. JACCARD DISEASE GRAPH
# ============================================================

X = A_ds
n_disease = X.shape[0]

intersection = X @ X.T
degrees = X.sum(axis=1)

union = degrees[:, None] + degrees[None, :] - intersection
J = np.divide(
    intersection,
    union,
    out=np.zeros_like(intersection, dtype=np.float32),
    where=union > 0
)

np.fill_diagonal(J, 0)

# Disease graph
G = nx.Graph()
G.add_nodes_from(range(n_disease))

rows, cols = np.where(np.triu(J, k=1) > 0)
for i, j in zip(rows, cols):
    G.add_edge(int(i), int(j), weight=float(J[i, j]))

print("Disease graph nodes:", G.number_of_nodes())
print("Disease graph edges:", G.number_of_edges())
print("Average degree:", np.mean([d for _, d in G.degree()]))

edge_rows = []
for i, j, data in G.edges(data=True):
    edge_rows.append({
        "disease_1": diseases[i],
        "disease_2": diseases[j],
        "jaccard_weight": data["weight"]
    })

disease_edges_df = pd.DataFrame(edge_rows)
disease_edges_df.to_csv(WORK_DIR / "weighted_disease_relationships.csv", index=False)

# Matrix export
pd.DataFrame(J, index=diseases, columns=diseases).to_csv(
    WORK_DIR / "jaccard_disease_relationship_matrix.csv"
)

display(disease_edges_df.sort_values("jaccard_weight", ascending=False).head(20))


In [ ]:
# ============================================================
# 7. GRAPH VISUALIZATION
# ============================================================

plt.figure(figsize=(13, 10))
pos = nx.spring_layout(G, seed=SEED, weight="weight")

weights = [G[u][v]["weight"] * 4 for u, v in G.edges()]
nx.draw_networkx_nodes(G, pos, node_size=450, alpha=0.85)
nx.draw_networkx_edges(G, pos, width=weights, alpha=0.25)
labels = {i: diseases[i][:18] for i in G.nodes()}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=7)

plt.title("Weighted Disease Relationship Graph from Symptom Topology",
          fontsize=16, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.savefig(WORK_DIR / "weighted_disease_graph.png", dpi=300, bbox_inches="tight")
plt.show()


## 8. Compact graph embedding

The paper describes 1-hop neighbourhood encoding, weighted-neighbour encoding and low-dimensional graph embedding. Here, the weighted adjacency matrix is normalized and compressed using truncated SVD. This gives a compact structural representation without creating a large descriptor matrix.


In [ ]:
# ============================================================
# 8. LOW-DIMENSIONAL GRAPH EMBEDDING
# ============================================================

A = J.copy().astype(np.float32)

# Weighted symmetric normalization
deg = A.sum(axis=1)
deg_inv_sqrt = 1.0 / np.sqrt(np.maximum(deg, 1e-8))
A_norm = deg_inv_sqrt[:, None] * A * deg_inv_sqrt[None, :]

# Combine one-hop topology and original symptom profile.
# This keeps the embedding symptom-topology guided.
base_features = np.hstack([
    X.astype(np.float32),
    A_norm.astype(np.float32)
])

embed_dim = min(32, max(2, base_features.shape[1] - 1), n_disease - 1)

svd = TruncatedSVD(n_components=embed_dim, random_state=SEED)
Z0 = svd.fit_transform(base_features).astype(np.float32)

print("Compact embedding:", Z0.shape)
print("Explained variance ratio:", float(svd.explained_variance_ratio_.sum()))

np.save(WORK_DIR / "compact_disease_embeddings.npy", Z0)
pd.DataFrame(Z0, index=diseases).to_csv(
    WORK_DIR / "compact_disease_embeddings.csv"
)


## 9. PyTorch lightweight adaptive local–global graph network

The implementation below follows the paper's architectural descriptions:

- single-layer graph message passing
- adaptive neighbour attention
- lightweight graph encoder
- low-rank graph attention
- compact global attention
- adaptive structural gating/fusion
- graph contrastive refinement
- neural bilinear decoder


In [ ]:
# ============================================================
# 9. PYTORCH INSTALL / IMPORT
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# ============================================================
# 10. MODEL COMPONENTS
# ============================================================

class AdaptiveNeighbourAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim, bias=False)
        self.k = nn.Linear(dim, dim, bias=False)
        self.v = nn.Linear(dim, dim, bias=False)
        self.scale = dim ** -0.5

    def forward(self, h, A):
        # h: [N,D], A: [N,N]
        q = self.q(h)
        k = self.k(h)
        v = self.v(h)

        scores = (q @ k.T) * self.scale

        # Structural weights are incorporated as log-priors.
        structural = torch.log(A.clamp_min(1e-8))
        scores = scores + structural

        mask = A > 0
        scores = scores.masked_fill(~mask, -1e9)

        alpha = torch.softmax(scores, dim=1)
        out = alpha @ v
        return out, alpha


class LowRankGraphAttention(nn.Module):
    def __init__(self, dim, rank=16):
        super().__init__()
        rank = min(rank, dim)
        self.u = nn.Linear(dim, rank, bias=False)
        self.v = nn.Linear(dim, rank, bias=False)
        self.proj = nn.Linear(rank, dim)

    def forward(self, h, A):
        # Low-rank global affinity.
        q = self.u(h)
        k = self.v(h)
        affinity = torch.sigmoid(q @ k.T / math.sqrt(q.shape[-1]))

        # Compact global structural prior.
        structural = A / (A.max() + 1e-8)
        affinity = 0.5 * affinity + 0.5 * structural

        weights = torch.softmax(affinity, dim=1)
        context = weights @ h
        return self.proj(context), weights


class LightweightGraphEncoder(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
            nn.Dropout(0.10),
            nn.Linear(hidden, hidden)
        )

    def forward(self, x):
        return self.net(x)


class AdaptiveStructuralFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Linear(dim, 1),
            nn.Sigmoid()
        )

    def forward(self, local, global_):
        gate = self.gate(torch.cat([local, global_], dim=1))
        fused = gate * local + (1 - gate) * global_
        return fused, gate


class LocalGlobalDiseaseNet(nn.Module):
    def __init__(self, in_dim, hidden=64, out_dim=48, rank=16):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.LayerNorm(hidden)
        )

        self.local_attn = AdaptiveNeighbourAttention(hidden)
        self.local_encoder = LightweightGraphEncoder(hidden, hidden)

        self.global_attn = LowRankGraphAttention(hidden, rank=rank)
        self.global_encoder = LightweightGraphEncoder(hidden, hidden)

        self.fusion = AdaptiveStructuralFusion(hidden)

        self.refine = nn.Sequential(
            nn.Linear(hidden, out_dim),
            nn.GELU(),
            nn.LayerNorm(out_dim)
        )

    def encode(self, x, A):
        h = self.input_proj(x)

        local_msg, local_alpha = self.local_attn(h, A)
        local = self.local_encoder(h + local_msg)

        global_msg, global_alpha = self.global_attn(h, A)
        global_ = self.global_encoder(h + global_msg)

        fused, gate = self.fusion(local, global_)
        z = F.normalize(self.refine(fused), dim=1)

        return z, {
            "local_attention": local_alpha,
            "global_attention": global_alpha,
            "fusion_gate": gate
        }

    def forward(self, x, A):
        return self.encode(x, A)


class BilinearDecoder(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.W = nn.Parameter(torch.empty(dim, dim))
        nn.init.xavier_uniform_(self.W)

    def forward(self, z):
        return z @ self.W @ z.T


class FullDiseaseRelationshipModel(nn.Module):
    def __init__(self, in_dim, hidden=64, emb_dim=48, rank=16):
        super().__init__()
        self.encoder = LocalGlobalDiseaseNet(
            in_dim, hidden=hidden, out_dim=emb_dim, rank=rank
        )
        self.decoder = BilinearDecoder(emb_dim)

    def encode(self, x, A):
        return self.encoder(x, A)

    def forward(self, x, A):
        z, aux = self.encode(x, A)
        logits = self.decoder(z)
        return logits, z, aux


## 10. Structure-preserving graph contrastive learning

Two views are generated:

1. neighbourhood-preserving augmentation
2. edge-weight-preserving augmentation

The contrastive loss encourages the same disease to remain close across the two graph views.


In [ ]:
# ============================================================
# 11. STRUCTURE-PRESERVING GRAPH AUGMENTATION + CONTRASTIVE LOSS
# ============================================================

def neighbourhood_preserving_augment(A, drop_rate=0.10):
    A2 = A.clone()
    n = A2.shape[0]

    # Randomly attenuate a small portion of existing edges.
    mask = (torch.rand_like(A2) < drop_rate) & (A2 > 0)
    upper = torch.triu(mask, diagonal=1)
    mask = upper | upper.T

    A2 = A2.masked_fill(mask, 0.0)
    A2 = torch.maximum(A2, A2.T)
    return A2

def edge_weight_preserving_augment(A, noise=0.05):
    A2 = A.clone()
    edge_mask = A2 > 0
    perturb = 1.0 + noise * torch.randn_like(A2)
    A2 = torch.where(edge_mask, A2 * perturb, A2)
    A2 = A2.clamp(min=0)
    A2 = torch.maximum(A2, A2.T)
    A2.fill_diagonal_(0)
    return A2

def info_nce(z1, z2, temperature=0.20):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    logits = z1 @ z2.T / temperature
    labels = torch.arange(z1.size(0), device=z1.device)

    loss12 = F.cross_entropy(logits, labels)
    loss21 = F.cross_entropy(logits.T, labels)

    return 0.5 * (loss12 + loss21)


## 11. Disease-pair sampling and supervised relationship learning

Positive pairs are the observed topology edges. Negative pairs are sampled from non-connected disease pairs. The neural bilinear decoder learns a disease relationship score.


In [ ]:
# ============================================================
# 12. PAIR SAMPLING
# ============================================================

def build_pair_labels(J, negative_ratio=2.0, seed=42):
    rng = np.random.default_rng(seed)
    n = J.shape[0]

    positives = np.argwhere(np.triu(J > 0, k=1))
    positive_pairs = [tuple(x) for x in positives]

    all_pairs = [(i, j) for i in range(n) for j in range(i+1, n)]
    positive_set = set(positive_pairs)
    negatives_pool = [p for p in all_pairs if p not in positive_set]

    n_neg = min(len(negatives_pool), int(len(positive_pairs) * negative_ratio))
    if n_neg > 0:
        idx = rng.choice(len(negatives_pool), size=n_neg, replace=False)
        negative_pairs = [negatives_pool[i] for i in idx]
    else:
        negative_pairs = []

    pairs = positive_pairs + negative_pairs
    labels = [1.0] * len(positive_pairs) + [0.0] * len(negative_pairs)

    order = rng.permutation(len(pairs))
    pairs = [pairs[i] for i in order]
    labels = np.array([labels[i] for i in order], dtype=np.float32)

    return np.array(pairs, dtype=np.int64), labels

pairs, pair_labels = build_pair_labels(J, negative_ratio=2.0, seed=SEED)

print("Total pairs:", len(pairs))
print("Positive:", int(pair_labels.sum()))
print("Negative:", int((1 - pair_labels).sum()))

# Pair-level train/validation split
idx = np.arange(len(pairs))
train_idx, val_idx = train_test_split(
    idx,
    test_size=0.20,
    random_state=SEED,
    stratify=pair_labels
)

train_pairs = pairs[train_idx]
train_labels = pair_labels[train_idx]
val_pairs = pairs[val_idx]
val_labels = pair_labels[val_idx]

print("Train pairs:", len(train_pairs))
print("Validation pairs:", len(val_pairs))


In [ ]:
# ============================================================
# 13. TRAINING
# ============================================================

feature_tensor = torch.tensor(base_features, dtype=torch.float32, device=DEVICE)
adj_tensor = torch.tensor(A, dtype=torch.float32, device=DEVICE)

model = FullDiseaseRelationshipModel(
    in_dim=base_features.shape[1],
    hidden=64,
    emb_dim=48,
    rank=16
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

train_pair_tensor = torch.tensor(train_pairs, dtype=torch.long, device=DEVICE)
train_y = torch.tensor(train_labels, dtype=torch.float32, device=DEVICE)

val_pair_tensor = torch.tensor(val_pairs, dtype=torch.long, device=DEVICE)
val_y = torch.tensor(val_labels, dtype=torch.float32, device=DEVICE)

history = {
    "epoch": [],
    "loss": [],
    "contrastive_loss": [],
    "relationship_loss": [],
    "val_auc": []
}

EPOCHS = 200
CONTRASTIVE_WEIGHT = 0.20

best_auc = -np.inf
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    # Main graph view
    logits, z, aux = model(feature_tensor, adj_tensor)

    # Pairwise scores
    pi = train_pair_tensor[:, 0]
    pj = train_pair_tensor[:, 1]
    pair_logits = logits[pi, pj]

    relationship_loss = F.binary_cross_entropy_with_logits(
        pair_logits, train_y
    )

    # Structure-preserving views
    A1 = neighbourhood_preserving_augment(adj_tensor, drop_rate=0.10)
    A2 = edge_weight_preserving_augment(adj_tensor, noise=0.04)

    z1, _ = model.encode(feature_tensor, A1)
    z2, _ = model.encode(feature_tensor, A2)

    contrastive_loss = info_nce(z1, z2)

    loss = relationship_loss + CONTRASTIVE_WEIGHT * contrastive_loss
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
    optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        val_logits, _, _ = model(feature_tensor, adj_tensor)
        val_scores = torch.sigmoid(
            val_logits[val_pair_tensor[:, 0], val_pair_tensor[:, 1]]
        ).detach().cpu().numpy()

    try:
        val_auc = roc_auc_score(val_labels, val_scores)
    except Exception:
        val_auc = np.nan

    history["epoch"].append(epoch)
    history["loss"].append(float(loss.item()))
    history["contrastive_loss"].append(float(contrastive_loss.item()))
    history["relationship_loss"].append(float(relationship_loss.item()))
    history["val_auc"].append(float(val_auc))

    if np.isfinite(val_auc) and val_auc > best_auc:
        best_auc = val_auc
        best_state = {k: v.detach().cpu().clone()
                      for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 20 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss.item():.4f} | "
            f"Rel {relationship_loss.item():.4f} | "
            f"CL {contrastive_loss.item():.4f} | "
            f"Val AUC {val_auc:.4f}"
        )

if best_state is not None:
    model.load_state_dict(best_state)

torch.save(model.state_dict(), WORK_DIR / "lightweight_adaptive_gnn.pt")
pd.DataFrame(history).to_csv(WORK_DIR / "training_history.csv", index=False)

print("Best validation AUC:", best_auc)


In [ ]:
# ============================================================
# 14. TRAINING CURVES
# ============================================================

hist_df = pd.DataFrame(history)

plt.figure(figsize=(11, 6))
plt.plot(hist_df["epoch"], hist_df["loss"], label="Total Loss", linewidth=2)
plt.plot(hist_df["epoch"], hist_df["relationship_loss"],
         label="Relationship Loss", linewidth=2)
plt.plot(hist_df["epoch"], hist_df["contrastive_loss"],
         label="Contrastive Loss", linewidth=2)
plt.xlabel("Epoch", fontsize=13, fontweight="bold")
plt.ylabel("Loss", fontsize=13, fontweight="bold")
plt.title("Training Loss", fontsize=16, fontweight="bold")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(WORK_DIR / "training_loss.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(11, 6))
plt.plot(hist_df["epoch"], hist_df["val_auc"], linewidth=2)
plt.xlabel("Epoch", fontsize=13, fontweight="bold")
plt.ylabel("Validation ROC-AUC", fontsize=13, fontweight="bold")
plt.title("Validation Relationship AUC", fontsize=16, fontweight="bold")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(WORK_DIR / "validation_auc.png", dpi=300, bbox_inches="tight")
plt.show()


## 12. Validation-based adaptive thresholding

The paper specifies validation-based adaptive thresholding, confidence-based edge selection and candidate edge discovery. The threshold below is selected on validation pairs using the F1 criterion, rather than being fixed arbitrarily.


In [ ]:
# ============================================================
# 15. VALIDATION-BASED ADAPTIVE THRESHOLD
# ============================================================

model.eval()

with torch.no_grad():
    full_logits, refined_z, aux = model(feature_tensor, adj_tensor)
    score_matrix = torch.sigmoid(full_logits).cpu().numpy()
    refined_embeddings = refined_z.cpu().numpy()

val_scores = score_matrix[val_pairs[:, 0], val_pairs[:, 1]]

thresholds = np.linspace(0.05, 0.95, 181)
threshold_rows = []

for t in thresholds:
    pred = (val_scores >= t).astype(int)
    threshold_rows.append({
        "threshold": t,
        "accuracy": accuracy_score(val_labels, pred),
        "precision": precision_score(val_labels, pred, zero_division=0),
        "recall": recall_score(val_labels, pred, zero_division=0),
        "f1": f1_score(val_labels, pred, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_rows)
best_row = threshold_df.loc[threshold_df["f1"].idxmax()]
adaptive_threshold = float(best_row["threshold"])

print("Adaptive threshold:", adaptive_threshold)
print(best_row)

threshold_df.to_csv(WORK_DIR / "validation_threshold_search.csv", index=False)


In [ ]:
# ============================================================
# 16. RELATIONSHIP EVALUATION
# ============================================================

val_pred = (val_scores >= adaptive_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    val_labels, val_pred, labels=[0, 1]
).ravel()

specificity = tn / max(tn + fp, 1)
sensitivity = tp / max(tp + fn, 1)
fpr = fp / max(fp + tn, 1)
fnr = fn / max(fn + tp, 1)

metrics = {
    "Accuracy": accuracy_score(val_labels, val_pred),
    "Precision": precision_score(val_labels, val_pred, zero_division=0),
    "Recall": recall_score(val_labels, val_pred, zero_division=0),
    "F1": f1_score(val_labels, val_pred, zero_division=0),
    "Specificity": specificity,
    "FPR": fpr,
    "FNR": fnr,
    "ROC_AUC": roc_auc_score(val_labels, val_scores),
    "PR_AUC": average_precision_score(val_labels, val_scores),
    "Adaptive_Threshold": adaptive_threshold
}

metrics_df = pd.DataFrame({
    "Metric": list(metrics.keys()),
    "Value": list(metrics.values())
})

display(metrics_df)
metrics_df.to_csv(WORK_DIR / "relationship_metrics.csv", index=False)


## 13. Enhanced disease relationship network

High-confidence predicted pairs are retained. Candidate edges are also reported separately. The original Jaccard graph is then augmented with newly inferred edges.


In [ ]:
# ============================================================
# 17. CONFIDENCE-BASED EDGE SELECTION + CANDIDATE DISCOVERY
# ============================================================

confidence_threshold = adaptive_threshold
candidate_threshold = max(0.50 * adaptive_threshold, 0.20)

enhanced = J.copy()

high_confidence_edges = []
candidate_edges = []

for i in range(n_disease):
    for j in range(i + 1, n_disease):
        score = float(score_matrix[i, j])

        if score >= confidence_threshold:
            high_confidence_edges.append({
                "disease_1": diseases[i],
                "disease_2": diseases[j],
                "predicted_score": score,
                "type": "high_confidence"
            })
            enhanced[i, j] = max(enhanced[i, j], score)
            enhanced[j, i] = enhanced[i, j]

        elif score >= candidate_threshold and J[i, j] == 0:
            candidate_edges.append({
                "disease_1": diseases[i],
                "disease_2": diseases[j],
                "predicted_score": score,
                "type": "candidate"
            })

high_conf_df = pd.DataFrame(high_confidence_edges)
candidate_df = pd.DataFrame(candidate_edges)

high_conf_df.to_csv(WORK_DIR / "high_confidence_relationships.csv", index=False)
candidate_df.to_csv(WORK_DIR / "candidate_relationships.csv", index=False)

pd.DataFrame(enhanced, index=diseases, columns=diseases).to_csv(
    WORK_DIR / "enhanced_disease_relationship_matrix.csv"
)

print("High-confidence inferred edges:", len(high_confidence_edges))
print("Candidate edges:", len(candidate_edges))

display(high_conf_df.sort_values("predicted_score", ascending=False).head(20))


In [ ]:
# ============================================================
# 18. FINAL ENHANCED GRAPH
# ============================================================

G_enhanced = nx.Graph()
G_enhanced.add_nodes_from(range(n_disease))

r, c = np.where(np.triu(enhanced > 0, k=1))
for i, j in zip(r, c):
    G_enhanced.add_edge(int(i), int(j), weight=float(enhanced[i, j]))

print("Original graph edges:", G.number_of_edges())
print("Enhanced graph edges:", G_enhanced.number_of_edges())

plt.figure(figsize=(14, 11))
pos = nx.spring_layout(G_enhanced, seed=SEED, weight="weight")
ew = [max(0.4, G_enhanced[u][v]["weight"] * 4)
      for u, v in G_enhanced.edges()]

nx.draw_networkx_nodes(
    G_enhanced, pos, node_size=420, alpha=0.85
)
nx.draw_networkx_edges(
    G_enhanced, pos, width=ew, alpha=0.20
)

labels = {i: diseases[i][:18] for i in G_enhanced.nodes()}
nx.draw_networkx_labels(
    G_enhanced, pos, labels=labels, font_size=7
)

plt.title(
    "Enhanced Disease Relationship Network",
    fontsize=17, fontweight="bold"
)
plt.axis("off")
plt.tight_layout()
plt.savefig(
    WORK_DIR / "enhanced_disease_relationship_network.png",
    dpi=300, bbox_inches="tight"
)
plt.show()

nx.write_gexf(
    G_enhanced,
    WORK_DIR / "enhanced_disease_relationship_network.gexf"
)


## 14. Resource mapping

According to the paper, supplementary files such as descriptions, medications, precautions, workouts and diets are connected **after** relationship discovery and are used for contextual resource presentation, not as predictive inputs to the neural network.

This cell automatically searches for these files if they are present.


In [ ]:
# ============================================================
# 19. SUPPLEMENTARY RESOURCE MAPPING
# ============================================================

resource_keywords = {
    "description": ["description", "descriptions"],
    "medication": ["medication", "medications", "drug", "drugs"],
    "precaution": ["precaution", "precautions"],
    "workout": ["workout", "workouts", "exercise"],
    "diet": ["diet", "diets", "food"]
}

resource_files = {}

search_roots = [WORK_DIR, Path("/content"), Path("/content/drive/MyDrive")]

for resource_type, kws in resource_keywords.items():
    matches = []
    for root in search_roots:
        if root.exists():
            for p in root.rglob("*.csv"):
                lname = p.name.lower()
                if any(k in lname for k in kws):
                    matches.append(p)
    # de-duplicate
    unique = []
    seen = set()
    for p in matches:
        try:
            rp = p.resolve()
            if rp not in seen:
                unique.append(rp)
                seen.add(rp)
        except:
            pass
    if unique:
        resource_files[resource_type] = unique[0]

print("Detected supplementary files:")
for k, v in resource_files.items():
    print(k, "->", v)

# Build a simple resource table using the first matching disease-like column.
resource_tables = {}

for rtype, path in resource_files.items():
    try:
        rdf = pd.read_csv(path)

        possible_disease = [
            c for c in rdf.columns
            if normalize_text(c) in {
                "disease", "disease name", "diseases",
                "prognosis", "label"
            }
        ]

        if possible_disease:
            dc = possible_disease[0]
        else:
            object_cols = rdf.select_dtypes(include=["object"]).columns.tolist()
            dc = object_cols[0] if object_cols else rdf.columns[0]

        rdf["_disease_key"] = rdf[dc].map(normalize_text)
        resource_tables[rtype] = rdf

    except Exception as e:
        print("Could not load", path, e)

print("Loaded resource tables:", list(resource_tables))


In [ ]:
# ============================================================
# 20. FINAL DISEASE RESOURCE TABLE
# ============================================================

final_resource = pd.DataFrame({"disease": diseases})
final_resource["_disease_key"] = final_resource["disease"].map(normalize_text)

for rtype, rdf in resource_tables.items():
    # Keep a compact representation of all non-key columns.
    dc = "_disease_key"
    value_cols = [c for c in rdf.columns if c != dc]
    if not value_cols:
        continue

    # Concatenate row information for each disease.
    tmp = rdf.groupby(dc)[value_cols].first().reset_index()

    # Rename columns to resource type where useful.
    tmp = tmp.rename(columns={
        c: f"{rtype}_{c}" for c in value_cols
    })

    final_resource = final_resource.merge(
        tmp, on="_disease_key", how="left"
    )

final_resource = final_resource.drop(columns=["_disease_key"])

final_resource.to_csv(
    WORK_DIR / "final_disease_relationship_resource_table.csv",
    index=False
)

display(final_resource.head(20))


## 15. Disease relationship query utility

This helper accepts a disease name and returns the strongest discovered relationships plus available mapped resources.


In [ ]:
# ============================================================
# 21. DISEASE RELATIONSHIP QUERY
# ============================================================

disease_to_idx = {normalize_text(d): i for i, d in enumerate(diseases)}

def query_disease(disease_name, top_k=10):
    key = normalize_text(disease_name)

    if key not in disease_to_idx:
        matches = [d for d in diseases if key in normalize_text(d)]
        if not matches:
            return None
        disease_name = matches[0]
        key = normalize_text(disease_name)

    i = disease_to_idx[key]

    scores = enhanced[i].copy()
    scores[i] = 0

    order = np.argsort(scores)[::-1]
    rows = []

    for j in order[:top_k]:
        if scores[j] > 0:
            rows.append({
                "query_disease": diseases[i],
                "related_disease": diseases[j],
                "relationship_strength": float(scores[j]),
                "original_jaccard": float(J[i, j]),
                "neural_score": float(score_matrix[i, j])
            })

    rel = pd.DataFrame(rows)

    resource = final_resource[
        final_resource["disease"].map(normalize_text) == key
    ]

    return rel, resource

# Example:
if diseases:
    result = query_disease(diseases[0], top_k=10)
    if result:
        display(result[0])
        display(result[1])


In [ ]:
# ============================================================
# 22. EXPORT ALL RESULTS TO ONE EXCEL WORKBOOK
# ============================================================

excel_path = WORK_DIR / "Symptom_Topology_GNN_Results.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    binary_df.to_excel(writer, sheet_name="Clean_Binary_Data", index=False)
    disease_edges_df.to_excel(writer, sheet_name="Jaccard_Edges", index=False)
    pd.DataFrame(J, index=diseases, columns=diseases).to_excel(
        writer, sheet_name="Jaccard_Matrix"
    )
    pd.DataFrame(Z0, index=diseases).to_excel(
        writer, sheet_name="Compact_Embeddings"
    )
    hist_df.to_excel(writer, sheet_name="Training_History", index=False)
    threshold_df.to_excel(writer, sheet_name="Threshold_Search", index=False)
    metrics_df.to_excel(writer, sheet_name="Metrics", index=False)
    high_conf_df.to_excel(writer, sheet_name="High_Confidence", index=False)
    candidate_df.to_excel(writer, sheet_name="Candidates", index=False)
    pd.DataFrame(enhanced, index=diseases, columns=diseases).to_excel(
        writer, sheet_name="Enhanced_Matrix"
    )
    final_resource.to_excel(
        writer, sheet_name="Resources", index=False
    )

print("Excel:", excel_path)


In [ ]:
# ============================================================
# 23. MODEL / ARTIFACT SUMMARY
# ============================================================

summary = {
    "num_diseases": int(n_disease),
    "num_symptoms": int(len(symptoms)),
    "bipartite_edges": int(B.number_of_edges()),
    "original_disease_edges": int(G.number_of_edges()),
    "enhanced_disease_edges": int(G_enhanced.number_of_edges()),
    "embedding_dimension": int(refined_embeddings.shape[1]),
    "validation_auc": float(metrics["ROC_AUC"]),
    "validation_pr_auc": float(metrics["PR_AUC"]),
    "adaptive_threshold": float(adaptive_threshold),
    "device": str(DEVICE)
}

summary_df = pd.DataFrame(
    {"Item": list(summary.keys()), "Value": list(summary.values())}
)
display(summary_df)

summary_df.to_csv(WORK_DIR / "implementation_summary.csv", index=False)


In [ ]:
# ============================================================
# 24. ZIP ALL OUTPUTS
# ============================================================

zip_path = Path("/content/Symptom_Topology_GNN_Implementation_Results.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in WORK_DIR.rglob("*"):
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(WORK_DIR)))

print("ZIP created:", zip_path)
print("Output directory:", WORK_DIR)


## Notes on reproducibility

- The implementation follows the stages described in the supplied paper.
- The paper specifies the conceptual operations but does not provide exact neural hyperparameters, train/validation protocol details, or exact implementation equations for every proposed module. Where those details are unspecified, this notebook uses explicit lightweight defaults rather than claiming they are values from the paper.
- The supplementary resource files are deliberately excluded from neural predictive inputs, consistent with the paper's Stage 12 description.
- Relationship metrics are computed from held-out disease-pair samples.
- The learned enhanced graph is exported as CSV and GEXF for downstream analysis.
